#Section 1 - Environment Setup

In [ ]:
import torch

print("PyTorch Version:", torch.__version__)
print("GPU Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))

PyTorch Version: 2.11.0+cu128
GPU Available: True
GPU Name: Tesla T4


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

project_dir = "/content/drive/MyDrive/RemoteSensing_Internship"
os.makedirs(project_dir, exist_ok=True)
print("Project directory:", project_dir)

Project directory: /content/drive/MyDrive/RemoteSensing_Internship


In [ ]:
%cd /content/drive/MyDrive/RemoteSensing_Internship

/content/drive/MyDrive/RemoteSensing_Internship


In [ ]:
!pip install -q timm kagglehub

#Section 2 - Dataset(NWPU-RESISC45)

In [ ]:
import kagglehub

path = kagglehub.dataset_download("aqibrehmanpirzada/nwpuresisc45")
print("Dataset downloaded to:", path)

100%|██████████| 408M/408M [00:23<00:00, 18.3MB/s]

Extracting files...


Dataset downloaded to: /root/.cache/kagglehub/datasets/aqibrehmanpirzada/nwpuresisc45/versions/1


In [ ]:
train_dir = os.path.join(path, "Dataset", "train", "train")
test_dir = os.path.join(path, "Dataset", "test", "test")

print("Train dir:", train_dir)
print("Test dir:", test_dir)

Train dir: /root/.cache/kagglehub/datasets/aqibrehmanpirzada/nwpuresisc45/versions/1/Dataset/train/train
Test dir: /root/.cache/kagglehub/datasets/aqibrehmanpirzada/nwpuresisc45/versions/1/Dataset/test/test


In [ ]:
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import ConcatDataset, DataLoader, random_split

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_part = ImageFolder(root=train_dir, transform=transform)
test_part = ImageFolder(root=test_dir, transform=transform)

assert train_part.classes == test_part.classes, "Class mismatch between train and test folders!"

dataset = ConcatDataset([train_part, test_part])
class_names = train_part.classes

print("Total Images:", len(dataset))
print("Number of Classes:", len(class_names))

Total Images: 31500
Number of Classes: 45


In [ ]:
dataset_size = len(dataset)
train_size = int(0.8 * dataset_size)
val_size = int(0.1 * dataset_size)
test_size = dataset_size - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    dataset,
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

print("Training Images:", len(train_dataset))
print("Validation Images:", len(val_dataset))
print("Testing Images:", len(test_dataset))

Training Images: 25200
Validation Images: 3150
Testing Images: 3150


In [ ]:
BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

images, labels = next(iter(train_loader))
print("Image Batch Shape:", images.shape)
print("Label Batch Shape:", labels.shape)

Image Batch Shape: torch.Size([32, 3, 224, 224])
Label Batch Shape: torch.Size([32])


#Section 3 - FractalNet Architecture


In [ ]:
import torch.nn as nn

class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)

In [ ]:
class FractalBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.path1 = ConvBlock(channels, channels)
        self.path2 = nn.Sequential(ConvBlock(channels, channels), ConvBlock(channels, channels))
        self.path3 = nn.Sequential(ConvBlock(channels, channels), ConvBlock(channels, channels), ConvBlock(channels, channels))

    def forward(self, x):
        p1 = self.path1(x)
        p2 = self.path2(x)
        p3 = self.path3(x)
        return (p1 + p2 + p3) / 3

In [ ]:
class FractalNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.stem = nn.Sequential(ConvBlock(3, 64), nn.MaxPool2d(2))
        self.layer1 = nn.Sequential(FractalBlock(64), nn.MaxPool2d(2))
        self.layer2 = nn.Sequential(ConvBlock(64, 128), FractalBlock(128), nn.MaxPool2d(2))
        self.layer3 = nn.Sequential(ConvBlock(128, 256), FractalBlock(256), nn.MaxPool2d(2))
        self.layer4 = nn.Sequential(ConvBlock(256, 512), FractalBlock(512))
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

In [ ]:
NUM_CLASSES = len(class_names)   # 45 for NWPU-RESISC45

model = FractalNet(num_classes=NUM_CLASSES)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")

Total Parameters: 20,387,181
Trainable Parameters: 20,387,181


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print(device)

cuda


#Section 4 - Training Setup

In [ ]:
criterion = nn.CrossEntropyLoss()
print(criterion)

CrossEntropyLoss()


In [ ]:
import torch.optim as optim

optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

print(optimizer)

AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.0001
    maximize: False
    weight_decay: 0.0001
)


In [ ]:
scheduler = optim.lr_scheduler.StepLR(
    optimizer,
    step_size=7,
    gamma=0.1
)

print(scheduler)

#Section 5 - Training Loop


In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = 100 * correct / total
    return epoch_loss, epoch_acc


def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = 100 * correct / total
    return epoch_loss, epoch_acc

In [ ]:
EPOCHS = 15

train_losses = []
train_accuracies = []
val_losses = []
val_accuracies = []

In [ ]:
best_accuracy = 0
import time

start_time = time.time()

for epoch in range(EPOCHS):

    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, device
    )

    val_loss, val_acc = evaluate(
        model, val_loader, criterion, device
    )

    scheduler.step()

    train_losses.append(train_loss)
    train_accuracies.append(train_acc)
    val_losses.append(val_loss)
    val_accuracies.append(val_acc)

    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"Train Loss : {train_loss:.4f}")
    print(f"Train Accuracy : {train_acc:.2f}%")
    print(f"Validation Loss : {val_loss:.4f}")
    print(f"Validation Accuracy : {val_acc:.2f}%")
    print("-"*40)

    if val_acc > best_accuracy:
        best_accuracy = val_acc
        torch.save(model.state_dict(), "best_fractalnet_nwpu45.pth")

end_time = time.time()
training_time = end_time - start_time

print(f"Training Time: {training_time:.2f} seconds")
print("Training Finished!")
print("Best Validation Accuracy:", best_accuracy)

Epoch 1/15
Train Loss : 2.2971
Train Accuracy : 36.96%
Validation Loss : 1.7705
Validation Accuracy : 49.84%
----------------------------------------
Epoch 2/15
Train Loss : 1.5956
Train Accuracy : 54.52%
Validation Loss : 1.4399
Validation Accuracy : 57.52%
----------------------------------------
Epoch 3/15
Train Loss : 1.2499
Train Accuracy : 63.47%
Validation Loss : 1.3082
Validation Accuracy : 61.30%
----------------------------------------
Epoch 4/15
Train Loss : 1.0224
Train Accuracy : 69.51%
Validation Loss : 1.0616
Validation Accuracy : 68.38%
----------------------------------------


#Section 6 - Evaluation

In [ ]:
model.load_state_dict(torch.load("best_fractalnet_nwpu45.pth"))
model.eval()
print("Best model loaded successfully!")

In [ ]:
test_loss, test_accuracy = evaluate(model, test_loader, criterion, device)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.2f}%")

In [ ]:
import torch.nn.functional as F

all_preds = []
all_labels = []
all_probs = []

model.eval()

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        probs = F.softmax(outputs, dim=1)
        preds = torch.argmax(probs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(20, 18))
sns.heatmap(
    cm,
    annot=False,
    fmt='d',
    cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names
)
plt.title("FractalNet — NWPU-RESISC45 Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(all_labels, all_preds, target_names=class_names))

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
import numpy as np

precision = precision_score(all_labels, all_preds, average='macro')
recall = recall_score(all_labels, all_preds, average='macro')
f1 = f1_score(all_labels, all_preds, average='macro')
auc = roc_auc_score(all_labels, np.array(all_probs), multi_class='ovr', average='macro')

print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")
print(f"AUC       : {auc:.4f}")

#Section 7 - Training Curves


In [ ]:
plt.figure(figsize=(8,5))
plt.plot(train_losses, marker='o', label="Train Loss")
plt.plot(val_losses, marker='o', label="Validation Loss")
plt.title("FractalNet NWPU-45 — Loss vs Epoch")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(train_accuracies, marker='o', label="Train Accuracy")
plt.plot(val_accuracies, marker='o', label="Validation Accuracy")
plt.title("FractalNet NWPU-45 — Accuracy vs Epoch")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.legend()
plt.show()

#Section 8 — Results Summary


In [ ]:
print("="*50)
print("FractalNet Results — NWPU-RESISC45")
print("="*50)
print(f"Training Time : {training_time:.2f} sec")
print(f"Training Loss : {train_losses[-1]:.4f}")
print(f"Validation Loss : {val_losses[-1]:.4f}")
print(f"Training Accuracy : {train_accuracies[-1]:.2f}%")
print(f"Validation Accuracy : {val_accuracies[-1]:.2f}%")
print(f"Test Accuracy : {test_accuracy:.2f}%")
print(f"Precision : {precision:.4f}")
print(f"Recall : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"AUC : {auc:.4f}")